In [5]:
import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Vipin\AppData\Roaming\nltk_data...


True

In [14]:
import pandas as pd
import numpy as np
import re
# Import required NLTK components
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from datetime import datetime

# --- 1. CONFIGURATION ---
NEWS_FILE_PATH = 'reddit_news.csv'
TARGET_FREQ = 'W' # Set to 'W' (Weekly) to match the frequency used in the XGBoost example
THRESHOLD = 0.2  # Threshold to classify sentiment: [-T, T] is Neutral

# --- 2. SENTIMENT LOGIC ---
def sentiment_to_direction(score):
    """
    Classifies the compound sentiment score into three directional categories.
    Assumes POSITIVE sentiment drives the base currency (USD) UP.
    """
    if score >= THRESHOLD:
        # High positive score -> USD likely to strengthen
        return 1
    elif score <= -THRESHOLD:
        # High negative score -> USD likely to weaken
        return -1
    else:
        # Score near zero -> Little effect
        return 0

def calculate_directional_sentiment_feature(file_path):
    """Loads news data, calculates sentiment, and aggregates to weekly frequency."""
    print("--- 1. Calculating Directional News Sentiment Feature ---")
    
    # Load Data
    try:
        df_news = pd.read_csv(file_path)
        # Display sample data for verification
        print("Sample raw news data:")
        print(df_news.head(2))
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return pd.DataFrame()

    # 1. Date Conversion and Indexing
    df_news.dropna(subset=['created_utc', 'title'], inplace=True)
    df_news['date'] = pd.to_datetime(df_news['created_utc'], unit='s')
    df_news.set_index('date', inplace=True)
    
    # 2. Sentiment Scoring (VADER)
    sia = SentimentIntensityAnalyzer()
    
    # Apply VADER scoring
    df_news['compound_score'] = df_news['title'].apply(
        lambda title: sia.polarity_scores(str(title))['compound']
    )
    
    # Optional: Filter news that mentions "USD", "Fed", or "US" to focus sentiment
    # For simplicity, we apply the classification to ALL news for now.
    
    # 3. Apply Directional Classification
    df_news['USD_Sentiment_Direction'] = df_news['compound_score'].apply(sentiment_to_direction)
    
    # 4. Weekly Aggregation
    # Since we need a single categorical feature for the week, we take the MODE (most frequent)
    # or the MEAN of the numerical score and re-classify. Let's use the MEAN score:
    
    # Aggregate the mean compound score for all news posts within the period.
    mean_sentiment = df_news['compound_score'].resample(TARGET_FREQ).mean()
    
    # Re-apply the directional classification to the aggregated mean score
    directional_feature = mean_sentiment.apply(sentiment_to_direction)
    directional_feature.name = 'News_Sentiment_Directional'
    
    df_sentiment = pd.DataFrame(directional_feature).dropna()
    
    print(f"\nDirectional sentiment feature calculated. Shape: {df_sentiment.shape}")
    print("Value Counts for the new feature:")
    print(df_sentiment['News_Sentiment_Directional'].value_counts())
    
    return df_sentiment

# --- 3. EXECUTION AND INTEGRATION ---

# 1. Generate the weekly directional sentiment feature
sentiment_df = calculate_directional_sentiment_feature(NEWS_FILE_PATH)

# 2. Integration: This feature must be joined with df_final and ONE-HOT ENCODED
# before being passed to XGBoost.

if not sentiment_df.empty:
    print("\n--- Integration Next Step ---")
    print("This feature needs to be ONE-HOT ENCODED before XGBoost training:")
    
    # 1. Perform One-Hot Encoding
    sentiment_encoded = pd.get_dummies(sentiment_df, columns=['News_Sentiment_Directional'], drop_first=True, prefix='Sent')
    print("Encoded Feature Columns:", sentiment_encoded.columns.tolist())
    
    # 2. Merge with your df_final (Example: Assuming df_final exists and is indexed weekly)
    # df_final = df_final.join(sentiment_encoded, how='left')
    # df_final.ffill(inplace=True)

--- 1. Calculating Directional News Sentiment Feature ---
Sample raw news data:
  subreddit                                              title body  score  \
0      news  Two leading Israeli human rights groups accuse...  NaN    307   
1      news  Texas county votes to release Uvalde school sh...  NaN    353   

                                                 url   created_utc  \
0  https://www.abc.net.au/news/2025-07-29/israeli...  1.753738e+09   
1  https://apnews.com/article/uvalde-school-shoot...  1.753738e+09   

   num_comments  
0            11  
1            16  

Directional sentiment feature calculated. Shape: (529, 1)
Value Counts for the new feature:
News_Sentiment_Directional
 0    350
-1    154
 1     25
Name: count, dtype: int64

--- Integration Next Step ---
This feature needs to be ONE-HOT ENCODED before XGBoost training:
Encoded Feature Columns: ['Sent_0', 'Sent_1']


In [17]:
sentiment_df

,News_Sentiment_Directional
date,
2015-06-21,-1
2015-06-28,1
2015-07-05,0
2015-07-12,0
2015-07-19,0
...,...
2025-07-06,0
2025-07-13,0
2025-07-20,0


In [16]:
sentiment_df.to_csv("sentiment_news.csv")